In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [13]:
file_path = "../synthetic_dataset_BETA_3.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1")

## EDA

In [14]:
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])
df['Sales_Date'] = pd.to_datetime(df['Sales_Date'])
df['Days_Since_Start'] = (df['Sales_Date'] - df['Purchase_Date']).dt.days

le_region = LabelEncoder()
df['Region'] = le_region.fit_transform(df['Region'])

le_season = LabelEncoder()
df['Season'] = le_season.fit_transform(df['Season'])

df.fillna(0, inplace=True)

scaler = StandardScaler()
df[['Quantity_Sold', 'Stock_Left', 'Returns', 'Discount', 'Days_Since_Start']] = scaler.fit_transform(
    df[['Quantity_Sold', 'Stock_Left', 'Returns', 'Discount', 'Days_Since_Start']])

X = df[['Days_Since_Start', 'Stock_Left', 'Returns', 'Region', 'Season', 'Holiday', 'Discount']]
y = df['Quantity_Sold']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13)

In [15]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=13)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

In [16]:
# Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=13)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)
gb_mae = mean_absolute_error(y_test, y_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))

In [17]:
# Подготовка данных для LSTM
X_train_lstm = np.expand_dims(X_train.values, axis=1)
X_test_lstm = np.expand_dims(X_test.values, axis=1)

# Создание LSTM модели
lstm_model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1, activation='linear')
])

# Компиляция и обучение
lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
lstm_model.fit(X_train_lstm, y_train, epochs=20, batch_size=32, validation_data=(X_test_lstm, y_test), verbose=1)

y_pred_lstm = lstm_model.predict(X_test_lstm)
lstm_mae = mean_absolute_error(y_test, y_pred_lstm)
lstm_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lstm))

C:\Users\Santa\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 1.0237 - val_loss: 0.9752
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0275 - val_loss: 0.9792
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0117 - val_loss: 0.9785
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0205 - val_loss: 0.9782
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0094 - val_loss: 0.9817
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9932 - val_loss: 0.9880
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.9944 - val_loss: 0.9806
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0088 - val_loss: 0.9791
Epoch 9/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0095 - val_loss: 0.9793
Epoch 10/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0186 - val_loss: 0.9843
Epoch 11/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0052 - val_loss: 0.9835
Epoch 12/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

In [18]:
# Вывод результатов
print(f"Random Forest - MAE: {rf_mae}, RMSE: {rf_rmse}")
print(f"Gradient Boosting - MAE: {gb_mae}, RMSE: {gb_rmse}")
print(f"LSTM - MAE: {lstm_mae}, RMSE: {lstm_rmse}")

Random Forest - MAE: 0.8163894044339918, RMSE: 1.0227499821860477
Gradient Boosting - MAE: 0.7910545449625769, RMSE: 0.9940077362350055
LSTM - MAE: 0.7882257554126221, RMSE: 0.9928539122070761


## LSTM наилучшее, будем его улучшать

1. Bidirectional LSTM — лучше улавливает временные зависимости.
2. Batch Normalization — стабилизирует обучение.
3. Дополнительные LSTM-слои — улучшенная обработка последовательности.
4. MinMaxScaler вместо StandardScaler — лучше для временных рядов.
5. Callbacks (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint) — автоматическая остановка, снижение learning rate, сохранение лучшей модели.
6. Оптимизатор RMSprop — лучше для рекуррентных сетей.

In [19]:
from keras.src.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.src.optimizers import RMSprop
from keras.src.layers import Bidirectional, BatchNormalization
from sklearn.preprocessing import MinMaxScaler

# Загрузка данных
file_path = "../synthetic_dataset_BETA_3.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1")

# Преобразование дат
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])
df['Sales_Date'] = pd.to_datetime(df['Sales_Date'])
df['Days_Since_Start'] = (df['Sales_Date'] - df['Purchase_Date']).dt.days

# Кодирование категориальных признаков
le_region = LabelEncoder()
df['Region'] = le_region.fit_transform(df['Region'])

le_season = LabelEncoder()
df['Season'] = le_season.fit_transform(df['Season'])

# Заполнение пропущенных значений
df.fillna(0, inplace=True)

# Масштабирование числовых данных
scaler = MinMaxScaler()
df[['Quantity_Sold', 'Stock_Left', 'Returns', 'Discount', 'Days_Since_Start']] = scaler.fit_transform(
    df[['Quantity_Sold', 'Stock_Left', 'Returns', 'Discount', 'Days_Since_Start']])

# Разделение данных на train/test
X = df[['Days_Since_Start', 'Stock_Left', 'Returns', 'Region', 'Season', 'Holiday', 'Discount']]
y = df['Quantity_Sold']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13)

# Подготовка данных для LSTM
X_train_lstm = np.expand_dims(X_train.values, axis=1)
X_test_lstm = np.expand_dims(X_test.values, axis=1)

# Создание улучшенной LSTM модели
lstm_model = Sequential([
    Bidirectional(LSTM(100, return_sequences=True, input_shape=(1, X_train.shape[1]))),
    BatchNormalization(),
    Dropout(0.2),
    LSTM(50, return_sequences=True),
    BatchNormalization(),
    Dropout(0.2),
    LSTM(25, return_sequences=False),
    Dropout(0.2),
    Dense(50, activation='relu'),
    Dense(1, activation='linear')
])

# Компиляция модели
lstm_model.compile(optimizer=RMSprop(learning_rate=0.001), loss='mse')

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
checkpoint = ModelCheckpoint("best_lstm_model.h5", save_best_only=True, monitor='val_loss')

# Обучение модели
lstm_model.fit(
    X_train_lstm, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_lstm, y_test),
    verbose=1,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)

# Оценка модели
y_pred_lstm = lstm_model.predict(X_test_lstm)
lstm_mae = mean_absolute_error(y_test, y_pred_lstm)
lstm_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lstm))

# Вывод результатов
print(f"LSTM - MAE: {lstm_mae}, RMSE: {lstm_rmse}")


Epoch 1/100


C:\Users\Santa\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


124/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0637

125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 0.0633 - val_loss: 0.0388 - learning_rate: 0.0010
Epoch 2/100
124/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0219

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0219 - val_loss: 0.0227 - learning_rate: 0.0010
Epoch 3/100
113/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0214

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0214 - val_loss: 0.0195 - learning_rate: 0.0010
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0204 - val_loss: 0.0200 - learning_rate: 0.0010
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0205 - val_loss: 0.0198 - learning_rate: 0.0010
Epoch 6/100
123/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0208

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0208 - val_loss: 0.0192 - learning_rate: 0.0010
Epoch 7/100
110/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0200

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0200 - val_loss: 0.0191 - learning_rate: 0.0010
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0202 - val_loss: 0.0195 - learning_rate: 0.0010
Epoch 9/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0198 - val_loss: 0.0191 - learning_rate: 0.0010
Epoch 10/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0199 - val_loss: 0.0196 - learning_rate: 0.0010
Epoch 11/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0205 - val_loss: 0.0194 - learning_rate: 0.0010
Epoch 12/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0200 - val_loss: 0.0193 - learning_rate: 0.0010
Epoch 13/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0197 - val_loss: 0.0192 - learning_rate: 5.0000e-04
Epoch 14/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0200 - val_loss: 0.0193 - learning_rate: 5.0000e-04
Epoch 15/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0197 - val_loss: 0.0192 - learning_rate: 5.0000

Модель	        MAE ↓ (чем меньше, тем лучше)	RMSE ↓ (чем меньше, тем лучше)

Ранее (LSTM)	0.788	                        0.993

Сейчас (LSTM)	0.110	                        0.138

In [21]:
from keras.src.layers import AdditiveAttention, LeakyReLU
from keras import Input, Model

# Загрузка данных
file_path = "../synthetic_dataset_BETA_3.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1")

# Преобразование дат
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])
df['Sales_Date'] = pd.to_datetime(df['Sales_Date'])
df['Days_Since_Start'] = (df['Sales_Date'] - df['Purchase_Date']).dt.days

# Кодирование категориальных признаков
le_region = LabelEncoder()
df['Region'] = le_region.fit_transform(df['Region'])

le_season = LabelEncoder()
df['Season'] = le_season.fit_transform(df['Season'])

# Генерация новых признаков
df['Rolling_Mean_7'] = df['Quantity_Sold'].rolling(window=7, min_periods=1).mean()
df['Rolling_Mean_30'] = df['Quantity_Sold'].rolling(window=30, min_periods=1).mean()
df['Lag_1'] = df['Quantity_Sold'].shift(1).fillna(0)
df.fillna(0, inplace=True)

# Масштабирование данных
scaler = MinMaxScaler()
df[['Quantity_Sold', 'Stock_Left', 'Returns', 'Discount', 'Days_Since_Start', 'Rolling_Mean_7', 'Rolling_Mean_30', 'Lag_1']] = scaler.fit_transform(
    df[['Quantity_Sold', 'Stock_Left', 'Returns', 'Discount', 'Days_Since_Start', 'Rolling_Mean_7', 'Rolling_Mean_30', 'Lag_1']])

# Разделение данных
X = df[['Days_Since_Start', 'Stock_Left', 'Returns', 'Region', 'Season', 'Holiday', 'Discount', 'Rolling_Mean_7', 'Rolling_Mean_30', 'Lag_1']]
y = df['Quantity_Sold']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13)

# Подготовка данных для LSTM
X_train_lstm = np.expand_dims(X_train.values, axis=1)
X_test_lstm = np.expand_dims(X_test.values, axis=1)

# Создание модели с Attention
input_layer = Input(shape=(1, X_train.shape[1]))
x = Bidirectional(LSTM(128, return_sequences=True))(input_layer)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

attention = AdditiveAttention()([x, x])
x = LSTM(64, return_sequences=True)(attention)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

x = LSTM(32, return_sequences=False)(x)
x = Dropout(0.2)(x)
x = Dense(64)(x)
x = LeakyReLU(alpha=0.1)(x)
output_layer = Dense(1, activation='linear')(x)

lstm_model = Model(inputs=input_layer, outputs=output_layer)

# Компиляция модели
lstm_model.compile(optimizer=Adam(learning_rate=0.0001, clipnorm=1.0), loss='mse')

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
checkpoint = ModelCheckpoint("../../old/pipeline/demand_forecasting_service_1/best_lstm_model_optimized.h5", save_best_only=True, monitor='val_loss')

# Обучение модели
lstm_model.fit(
    X_train_lstm, y_train,
    epochs=150,
    batch_size=128,
    validation_data=(X_test_lstm, y_test),
    verbose=1,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)

# Оценка модели
y_pred_lstm = lstm_model.predict(X_test_lstm)
lstm_mae = mean_absolute_error(y_test, y_pred_lstm)
lstm_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lstm))

# Вывод результатов
print(f"Optimized LSTM with Attention - MAE: {lstm_mae}, RMSE: {lstm_rmse}")

Epoch 1/150


C:\Users\Santa\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
C:\Users\Santa\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\ops\nn.py:907: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


55/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1775

C:\Users\Santa\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\ops\nn.py:907: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 0.1710 - val_loss: 0.1732 - learning_rate: 1.0000e-04
Epoch 2/150
60/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0494

...

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0215 - val_loss: 0.0167 - learning_rate: 1.0000e-04
Epoch 17/150
61/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0203

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0204 - val_loss: 0.0166 - learning_rate: 1.0000e-04
Epoch 18/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0204 - val_loss: 0.0166 - learning_rate: 1.0000e-04
Epoch 19/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0207 - val_loss: 0.0166 - learning_rate: 1.0000e-04
Epoch 20/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0203 - val_loss: 0.0166 - learning_rate: 1.0000e-04
Epoch 21/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0200 - val_loss: 0.0166 - learning_rate: 1.0000e-04
Epoch 22/150
56/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0203

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0202 - val_loss: 0.0165 - learning_rate: 1.0000e-04
Epoch 23/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0203 - val_loss: 0.0166 - learning_rate: 5.0000e-05
Epoch 24/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0194 - val_loss: 0.0166 - learning_rate: 5.0000e-05
Epoch 25/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0199 - val_loss: 0.0167 - learning_rate: 5.0000e-05
Epoch 26/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0196 - val_loss: 0.0168 - learning_rate: 5.0000e-05
Epoch 27/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0199 - val_loss: 0.0168 - learning_rate: 5.0000e-05
Epoch 28/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0193 - val_loss: 0.0169 - learning_rate: 2.5000e-05
Epoch 29/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0192 - val_loss: 0.0168 - learning_rate: 2.5000e-05
Epoch 30/150
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0191 - val_loss: 0.0168 - learning_rate

C:\Users\Santa\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\ops\nn.py:907: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
Optimized LSTM with Attention - MAE: 0.10186883695423603, RMSE: 0.12851621986038025
